# OHLCV Fetcher
Downloads historical daily price data for the full ticker universe (Alpha Vantage) and macro instruments (yfinance).

| Destination | Source | Tickers |
|---|---|---|
| `data/raw_price/` | Alpha Vantage TIME_SERIES_DAILY_ADJUSTED | `TICKERS` + `BENCHMARK_TICKER` |
| `data/raw_macro/` | yfinance | `MACRO_YFINANCE` |

## Cell 1 — Setup & Imports

In [1]:
import os
import time

import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv

from config import BENCHMARK_TICKER, MACRO_YFINANCE, TICKERS

# ---------------------------------------------------------------------------
# Environment
# ---------------------------------------------------------------------------
load_dotenv()
AV_API_KEY: str = os.environ["ALPHA_VANTAGE_API_KEY"]

# ---------------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------------
RAW_PRICE_DIR: str = "data/raw_price"
RAW_MACRO_DIR: str = "data/raw_macro"

os.makedirs(RAW_PRICE_DIR, exist_ok=True)
os.makedirs(RAW_MACRO_DIR, exist_ok=True)

print(f"Ticker universe : {len(TICKERS)} tickers + {len(BENCHMARK_TICKER)} benchmark")
print(f"Macro instruments : {MACRO_YFINANCE}")
print(f"Output dirs ready : {RAW_PRICE_DIR}  |  {RAW_MACRO_DIR}")

Ticker universe : 50 tickers + 1 benchmark
Macro instruments : ['ES=F', 'NQ=F', '^TNX']
Output dirs ready : data/raw_price  |  data/raw_macro


## Cell 2 — yfinance Macro Fetcher
Downloads 5 years of daily OHLCV data for each instrument in `MACRO_YFINANCE` and saves to `data/raw_macro/`.

In [2]:
def fetch_macro_yfinance(ticker: str, output_dir: str, period: str = "5y") -> None:
    """
    Download daily OHLCV history for a single macro instrument via yfinance
    and persist it as a CSV.

    Parameters
    ----------
    ticker : str
        yfinance-compatible ticker symbol (e.g. 'ES=F', '^TNX').
    output_dir : str
        Directory in which to save the CSV file.
    period : str
        Lookback period string accepted by yfinance (default '5y').
    """
    # Sanitise ticker for use as a filename ('^TNX' → '_TNX')
    safe_name: str = ticker.replace("^", "_").replace("=", "_")
    out_path: str = os.path.join(output_dir, f"{safe_name}_daily.csv")

    df: pd.DataFrame = yf.download(
        ticker,
        period=period,
        interval="1d",
        auto_adjust=True,
        progress=False,
    )

    if df.empty:
        print(f"  [WARN] No data returned for {ticker} — skipping.")
        return

    df.to_csv(out_path)
    print(f"  [OK] {ticker:>8s}  →  {out_path}  ({len(df)} rows)")


print("Fetching macro data via yfinance...")
for macro_ticker in MACRO_YFINANCE:
    fetch_macro_yfinance(macro_ticker, RAW_MACRO_DIR)

print("\nMacro fetch complete.")

Fetching macro data via yfinance...
  [OK]     ES=F  →  data/raw_macro/ES_F_daily.csv  (1259 rows)
  [OK]     NQ=F  →  data/raw_macro/NQ_F_daily.csv  (1259 rows)
  [OK]     ^TNX  →  data/raw_macro/_TNX_daily.csv  (1256 rows)

Macro fetch complete.


## Cell 3 — Alpha Vantage Price Fetcher
Iterates through `TICKERS + BENCHMARK_TICKER`, pulls `TIME_SERIES_DAILY_ADJUSTED` with `outputsize=full` and `datatype=csv`, and handles rate-limit responses with an automatic 65-second back-off + retry.

In [3]:
AV_BASE_URL: str = "https://www.alphavantage.co/query"
INTER_REQUEST_DELAY: float = 0.85  # seconds between successful requests
AV_RATE_LIMIT_PHRASES: list[str] = [
    "api call frequency",
    "rate limit",
    "thank you for using alpha vantage",  # appears in the JSON note field
    "standard api call frequency",
]


def _is_rate_limited(response_text: str) -> bool:
    """
    Return True if the Alpha Vantage response body contains a rate-limit
    or API-call-frequency error message.

    Parameters
    ----------
    response_text : str
        Raw text body of the HTTP response.

    Returns
    -------
    bool
    """
    lowered: str = response_text.lower()
    return any(phrase in lowered for phrase in AV_RATE_LIMIT_PHRASES)


def fetch_av_daily_adjusted(
    ticker: str,
    api_key: str,
    output_dir: str,
    max_retries: int = 3,
) -> None:
    """
    Fetch full daily-adjusted OHLCV history from Alpha Vantage for one ticker
    and save the raw CSV response to disk.

    Retries up to `max_retries` times after a 65-second sleep whenever a
    rate-limit response is detected.

    Parameters
    ----------
    ticker : str
        Equity ticker symbol (e.g. 'AAPL').
    api_key : str
        Alpha Vantage API key.
    output_dir : str
        Directory in which to write the CSV file.
    max_retries : int
        Maximum number of retry attempts on rate-limit errors (default 3).
    """
    params: dict[str, str] = {
        "function": "TIME_SERIES_DAILY_ADJUSTED",
        "symbol": ticker,
        "outputsize": "full",
        "datatype": "csv",
        "apikey": api_key,
    }
    out_path: str = os.path.join(output_dir, f"{ticker}_daily.csv")

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(AV_BASE_URL, params=params, timeout=30)
            response.raise_for_status()
        except requests.exceptions.RequestException as exc:
            print(f"  [ERROR] {ticker} — network error on attempt {attempt}: {exc}")
            if attempt < max_retries:
                time.sleep(65)
            continue

        if _is_rate_limited(response.text):
            print(
                f"  [RATE LIMIT] {ticker} — attempt {attempt}/{max_retries}. "
                "Sleeping 65 s before retry..."
            )
            time.sleep(65)
            continue

        # Successful response — persist raw CSV text
        with open(out_path, "w", encoding="utf-8") as fh:
            fh.write(response.text)

        print(f"  [OK] {ticker:>6s}  →  {out_path}")
        return

    print(f"  [FAILED] {ticker} — exhausted {max_retries} retries. Skipping.")


# ---------------------------------------------------------------------------
# Main fetch loop: 50 universe tickers + SPY benchmark
# ---------------------------------------------------------------------------
all_price_tickers: list[str] = TICKERS + BENCHMARK_TICKER
total: int = len(all_price_tickers)

print(f"Fetching daily-adjusted OHLCV for {total} tickers via Alpha Vantage...\n")

for idx, ticker in enumerate(all_price_tickers, start=1):
    print(f"[{idx:>3}/{total}] {ticker}", end="  ")
    fetch_av_daily_adjusted(ticker, AV_API_KEY, RAW_PRICE_DIR)
    time.sleep(INTER_REQUEST_DELAY)

print("\nAlpha Vantage price fetch complete.")

Fetching daily-adjusted OHLCV for 51 tickers via Alpha Vantage...

[  1/51] NVDA    [OK]   NVDA  →  data/raw_price/NVDA_daily.csv
[  2/51] AMD    [OK]    AMD  →  data/raw_price/AMD_daily.csv
[  3/51] TSM    [OK]    TSM  →  data/raw_price/TSM_daily.csv
[  4/51] AVGO    [OK]   AVGO  →  data/raw_price/AVGO_daily.csv
[  5/51] MU    [OK]     MU  →  data/raw_price/MU_daily.csv
[  6/51] INTC    [OK]   INTC  →  data/raw_price/INTC_daily.csv
[  7/51] ARM    [OK]    ARM  →  data/raw_price/ARM_daily.csv
[  8/51] QCOM    [OK]   QCOM  →  data/raw_price/QCOM_daily.csv
[  9/51] ASML    [OK]   ASML  →  data/raw_price/ASML_daily.csv
[ 10/51] SMCI    [OK]   SMCI  →  data/raw_price/SMCI_daily.csv
[ 11/51] MRVL    [OK]   MRVL  →  data/raw_price/MRVL_daily.csv
[ 12/51] TXN    [OK]    TXN  →  data/raw_price/TXN_daily.csv
[ 13/51] KLAC    [OK]   KLAC  →  data/raw_price/KLAC_daily.csv
[ 14/51] AMAT    [OK]   AMAT  →  data/raw_price/AMAT_daily.csv
[ 15/51] LRCX    [OK]   LRCX  →  data/raw_price/LRCX_daily.csv
